# Preprocessing
## Column Mapping, Missing Values & Duplicates

## Loading the Dataset

In [1]:
import pandas as pd
import pickle

df = pd.read_csv("data/cs-training.csv")
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])
print("Original shape:", df.shape)
print("Original columns:", df.columns.tolist())

Original shape: (150000, 11)
Original columns: ['SeriousDlqin2yrs', 'RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents']


## Column Mapping

In [2]:
column_map = {
    "SeriousDlqin2yrs":                     "Target",
    "RevolvingUtilizationOfUnsecuredLines":  "RevolvingUtilization",
    "age":                                   "Age",
    "NumberOfTime30-59DaysPastDueNotWorse":  "Times30_59Late",
    "DebtRatio":                             "DebtRatio",
    "MonthlyIncome":                         "MonthlyIncome",
    "NumberOfOpenCreditLinesAndLoans":       "OpenCreditLines",
    "NumberOfTimes90DaysLate":               "Times90Late",
    "NumberRealEstateLoansOrLines":          "RealEstateLines",
    "NumberOfTime60-89DaysPastDueNotWorse":  "Times60_89Late",
    "NumberOfDependents":                    "Dependents",
}
df = df.rename(columns=column_map)
print("Column name mapping applied:")
for old, new in column_map.items():
    print(f"  {old:<45} -> {new}")
print(f"\nNew columns: {df.columns.tolist()}")

Column name mapping applied:
  SeriousDlqin2yrs                              -> Target
  RevolvingUtilizationOfUnsecuredLines          -> RevolvingUtilization
  age                                           -> Age
  NumberOfTime30-59DaysPastDueNotWorse          -> Times30_59Late
  DebtRatio                                     -> DebtRatio
  MonthlyIncome                                 -> MonthlyIncome
  NumberOfOpenCreditLinesAndLoans               -> OpenCreditLines
  NumberOfTimes90DaysLate                       -> Times90Late
  NumberRealEstateLoansOrLines                  -> RealEstateLines
  NumberOfTime60-89DaysPastDueNotWorse          -> Times60_89Late
  NumberOfDependents                            -> Dependents

New columns: ['Target', 'RevolvingUtilization', 'Age', 'Times30_59Late', 'DebtRatio', 'MonthlyIncome', 'OpenCreditLines', 'Times90Late', 'RealEstateLines', 'Times60_89Late', 'Dependents']


## Missing Values

In [3]:
print("Missing values before handling:")
print(df.isnull().sum())
missing_cols = df.columns[df.isnull().any()]
print("\nMean, Median, and Mode for columns with missing values:")
for col in missing_cols:
    print(f"\n  Column: {col}")
    print(f"  Missing values : {df[col].isnull().sum()} ({df[col].isnull().mean() * 100:.2f}%)")
    print(f"  Mean           : {df[col].mean():.2f}")
    print(f"  Median         : {df[col].median():.2f}")
    print(f"  Mode           : {df[col].mode().tolist()}")
    print(f"  Std Dev        : {df[col].std():.2f}")
    print(f"  Skewness       : {df[col].skew():.2f}")

Missing values before handling:
Target                      0
RevolvingUtilization        0
Age                         0
Times30_59Late              0
DebtRatio                   0
MonthlyIncome           29731
OpenCreditLines             0
Times90Late                 0
RealEstateLines             0
Times60_89Late              0
Dependents               3924
dtype: int64

Mean, Median, and Mode for columns with missing values:

  Column: MonthlyIncome
  Missing values : 29731 (19.82%)
  Mean           : 6670.22
  Median         : 5400.00
  Mode           : [5000.0]
  Std Dev        : 14384.67
  Skewness       : 114.04

  Column: Dependents
  Missing values : 3924 (2.62%)
  Mean           : 0.76
  Median         : 0.00
  Mode           : [0.0]
  Std Dev        : 1.12
  Skewness       : 1.59


In [4]:
print("--- Imputation Decision ---")
print("Using MEDIAN for both columns.")
print("Reason: Both features are right-skewed (skewness > 0),")
print("so the mean is inflated by outliers. Median is more robust.")

# Save training medians for test set later
train_medians = {
    "MonthlyIncome": df["MonthlyIncome"].median(),
    "Dependents": df["Dependents"].median(),
    "Age": df["Age"].median()
}
print(f"\nTraining medians: {train_medians}")

df["MonthlyIncome"] = df["MonthlyIncome"].fillna(train_medians["MonthlyIncome"])
df["Dependents"] = df["Dependents"].fillna(train_medians["Dependents"])

invalid_age = (df["Age"] == 0).sum()
if invalid_age > 0:
    print(f"\nFound {invalid_age} row(s) with Age = 0 (invalid)")
    print(f"Replacing with median Age: {train_medians['Age']:.0f}")
    df.loc[df["Age"] == 0, "Age"] = train_medians["Age"]

print("\nMissing values after handling:")
print(df.isnull().sum())

--- Imputation Decision ---
Using MEDIAN for both columns.
Reason: Both features are right-skewed (skewness > 0),
so the mean is inflated by outliers. Median is more robust.

Training medians: {'MonthlyIncome': np.float64(5400.0), 'Dependents': np.float64(0.0), 'Age': np.float64(52.0)}

Found 1 row(s) with Age = 0 (invalid)
Replacing with median Age: 52

Missing values after handling:
Target                  0
RevolvingUtilization    0
Age                     0
Times30_59Late          0
DebtRatio               0
MonthlyIncome           0
OpenCreditLines         0
Times90Late             0
RealEstateLines         0
Times60_89Late          0
Dependents              0
dtype: int64


## Duplicate Rows

In [5]:
print("Duplicate rows before handling:", df.duplicated().sum())
df = df.drop_duplicates()
print("Shape after removing duplicates:", df.shape)
print("Duplicate rows after handling:", df.duplicated().sum())

Duplicate rows before handling: 767


Shape after removing duplicates: (149233, 11)


Duplicate rows after handling: 0


## Target Distribution

In [6]:
target_counts = df["Target"].value_counts()
print(f"  No Default (0): {target_counts[0]:>7,} ({target_counts[0]/len(df)*100:.1f}%)")
print(f"  Default    (1): {target_counts[1]:>7,} ({target_counts[1]/len(df)*100:.1f}%)")
print(f"  Imbalance ratio: {target_counts[0] / target_counts[1]:.1f} : 1")

  No Default (0): 139,229 (93.3%)
  Default    (1):  10,004 (6.7%)
  Imbalance ratio: 13.9 : 1


## Save

In [7]:
df.to_csv("outputs/cleaned.csv", index=False)
print(f"Saved: outputs/cleaned.csv ({df.shape[0]:,} rows x {df.shape[1]} cols)")

# Save training medians for test_preprocessing.ipynb
with open("outputs/train_medians.pkl", "wb") as f:
    pickle.dump(train_medians, f)
print("Saved: outputs/train_medians.pkl")

Saved: outputs/cleaned.csv (149,233 rows x 11 cols)
Saved: outputs/train_medians.pkl
